### ElasticNet Regression

Ridge ve Lasso'nun her ikisinin de bir zayıf noktası vardır:
- Ridge: Katsayıları küçültür ama hiç eleme yapmaz (gereksiz değişkenler hep kalır)
- Lasso: Eleme yapar ama çok korelasyonlu değişkenler arasında rastgele/tutarsız bir seçim yapabilir

ElasticNet, ikisini birleştirir:
- Loss = Hata + λ1 × Σ|w| + λ2 × Σ(w²)

sklearn'ün kullandığı parametrelendirmeyle:
- Loss = Hata + α × [ r × Σ|w| + (1-r) × Σ(w²) ]

- α (alpha) → toplam ceza gücü
- r (sklearn'de l1_ratio) → cezanın ne kadarının L1 (Lasso), ne kadarının L2 (Ridge) olacağını belirleyen oran (0 ile 1 arası)
  - l1_ratio=1 → Saf Lasso
  - l1_ratio=0 → Saf Ridge
  - l1_ratio=0.5 → Yarı yarıya karışım

### Neden Kullanılır?

Özellikle çok sayıda, birbiriyle korelasyonlu değişken olduğunda, ElasticNet hem gereksiz değişkenleri eler (Lasso özelliği) hem de kalan değişkenler arasında daha stabil/dengeli katsayı dağılımı sağlar (Ridge özelliği).

Not: ElasticNet'in Lasso veya Ridge'den "kesin olarak" daha iyi sonuç vereceği garanti değildir, bu veri setinin yapısına bağlıdır. Gücü özellikle çok yüksek boyutlu (onlarca/yüzlerce değişkenli) ve birbirine yüksek korelasyonlu gruplar halinde değişkenlerin olduğu veri setlerinde daha belirgin ortaya çıkar.

In [2]:
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1. Veri setini yükle ve temizle
df = sns.load_dataset('mpg')
df = df.dropna(subset=['horsepower'])

# ============================================
# SENARYO 1: Polynomial (degree=3) ile ElasticNet
# ============================================
poly3 = PolynomialFeatures(degree=3)
X_poly3 = poly3.fit_transform(df[['horsepower']])
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(X_poly3, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=== Senaryo 1: Polynomial (degree=3) ===")
for alpha_deger in [0.01, 0.1, 1, 10]:
    for l1_oran in [0.1, 0.5, 0.9]:
        model = ElasticNet(alpha=alpha_deger, l1_ratio=l1_oran)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        r2 = r2_score(y_test, y_pred)
        print(f"Alpha={alpha_deger}, l1_ratio={l1_oran}: R²={r2:.4f}, Katsayılar = {model.coef_}")

# ============================================
# SENARYO 2: Multicollinearity'li Değişkenler (weight, horsepower, displacement)
# ============================================
X_multi = df[['horsepower', 'weight', 'displacement']]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_multi, y, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_test2_scaled = scaler2.transform(X_test2)

print("\n=== Senaryo 2: Multicollinearity (horsepower, weight, displacement) ===")
for alpha_deger in [0.01, 0.1, 1, 10]:
    for l1_oran in [0.1, 0.5, 0.9]:
        model = ElasticNet(alpha=alpha_deger, l1_ratio=l1_oran)
        model.fit(X_train2_scaled, y_train2)
        y_pred = model.predict(X_test2_scaled)
        r2 = r2_score(y_test2, y_pred)
        print(f"Alpha={alpha_deger}, l1_ratio={l1_oran}: R²={r2:.4f}, Katsayılar = {model.coef_}")

=== Senaryo 1: Polynomial (degree=3) ===
Alpha=0.01, l1_ratio=0.1: R²=0.6452, Katsayılar = [  0.         -10.78966395  -0.63997062   5.64623463]
Alpha=0.01, l1_ratio=0.5: R²=0.6438, Katsayılar = [  0.         -11.54789865   0.           5.74555766]
Alpha=0.01, l1_ratio=0.9: R²=0.6412, Katsayılar = [  0.         -12.66033186   1.26108445   5.5663721 ]
Alpha=0.1, l1_ratio=0.1: R²=0.6023, Katsayılar = [ 0.         -5.76043733 -1.60322997  1.75675933]
Alpha=0.1, l1_ratio=0.5: R²=0.6207, Katsayılar = [ 0.         -7.13437778 -0.90319583  2.35720808]
Alpha=0.1, l1_ratio=0.9: R²=0.6429, Katsayılar = [ 0.         -9.72407021 -0.          3.93728301]
Alpha=1, l1_ratio=0.1: R²=0.5110, Katsayılar = [ 0.         -2.05376938 -1.43608179 -0.85292104]
Alpha=1, l1_ratio=0.5: R²=0.5288, Katsayılar = [ 0.         -2.53957515 -1.47745652 -0.48238924]
Alpha=1, l1_ratio=0.9: R²=0.5738, Katsayılar = [ 0.         -4.67790568 -0.17248194 -0.        ]
Alpha=10, l1_ratio=0.1: R²=0.2066, Katsayılar = [ 0.       

### ElasticNet Sonuçları — İki Senaryo Karşılaştırması

**Senaryo 1 (Polynomial, degree=3):** En iyi sonuç Alpha=0.01, l1_ratio=0.1'de R²=0.6452 — Ridge (0.6423) ve Lasso'nun (0.6453) en iyi sonuçlarına çok yakın. Bu basit 3 özellikli senaryoda ElasticNet'in avantajı belirgin değil.

**Senaryo 2 (Multicollinearity: horsepower, weight, displacement):** En iyi sonuç Alpha=1, l1_ratio=0.9'da R²=0.6673 — bu, hem sadece weight kullanan tek değişkenli modelden (R²=0.6533) hem de cezasız 3 değişkenli modelden (R²=0.6471) daha iyi! Dikkat çekici nokta: bu en iyi sonuçta hiçbir katsayı sıfırlanmadı, ElasticNet üç değişkeni de tuttu ama katsayıları dengeleyerek multicollinearity'nin zararını azalttı.

**Genel ders:** ElasticNet'in gerçek gücü, çok sayıda korelasyonlu değişkenin olduğu senaryolarda ortaya çıkıyor — basit, az değişkenli senaryolarda Ridge/Lasso'dan belirgin bir farkı olmayabilir. Her iki senaryoda da alpha=10 civarında aşırı düzenlileştirme modeli işlevsizleştirdi (R² negatife düştü).